# 01 · MSX-Hawkes: a certified global MLE

Simulate a 2-d phase-type Hawkes process with a news shock, fit MSX, and read the Fenchel duality-gap certificate (Proposition 2).

In [1]:
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")  # notebooks run on CPU
os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
import numpy as np, json, warnings
warnings.filterwarnings("ignore")

In [2]:
from infodiff.models.dictionary import PhaseTypeDictionary
from infodiff.models.features import DesignSpec
from infodiff.models.msx import MSXHawkes
from infodiff.sim.cluster import HawkesTruth, simulate
endo = PhaseTypeDictionary.log_grid(0.05, 5.0, 3, orders=2)
A = np.zeros((2, 2, endo.K, endo.R)); A[0, 0, 0, 1] = 0.3; A[1, 1, 2, 0] = 0.4; A[0, 1, 1, 0] = 0.2
truth = HawkesTruth(endo=endo, A=A, mu=np.array([1.0, 0.7]))
t0 = np.arange(20) * 3000.0
data = simulate(truth, t0=t0, t1=t0 + 2500.0, burn=50.0, seed=3)
print("events per dim:", data.counts())

events per dim: [88706 58855]


In [3]:
m = MSXHawkes(DesignSpec(endo=endo), device="cpu", gap_tol=1e-6, gap_rel=0.0).fit(data)
for r in m.reports:
    print(f"dim {r.dim}: loglik {r.loglik:.3f}  certified gap {r.gap:.2e} nats  E[N]/N = {r.expected_count / r.n_events:.6f}")
print("true branching matrix\n", A.sum(axis=(2, 3)))
print("estimated\n", m.branching_matrix().round(3))

dim 0: loglik -24287.034  certified gap 2.82e-08 nats  E[N]/N = 1.000000
dim 1: loglik -48575.600  certified gap 2.87e-08 nats  E[N]/N = 1.000000
true branching matrix
 [[0.3 0.2]
 [0.  0.4]]
estimated
 [[0.308 0.195]
 [0.    0.395]]
